# Get Conservation and ENCODE from UCSC

In [1]:
import json, urllib.request
from pathlib import Path

PROJECT_ROOT = Path("../").resolve()
OUT_DIR = PROJECT_ROOT / "tracks" / "ucsc_raw"

GENOME = "hg38"
chrom, start1, end1 = "chr19", 11_039_463, 11_163_820   # 1-based inclusive, like the browser

In [2]:
import json, urllib.request

def ucsc_items(track, chrom, start1, end1, genome="hg38"):
    """Items of a UCSC track overlapping a 1-based inclusive region."""
    url = (f"https://api.genome.ucsc.edu/getData/track?genome={genome};track={track}"
           f";chrom={chrom};start={start1 - 1};end={end1}")
    data = json.load(urllib.request.urlopen(url))[track]
    return data[chrom] if isinstance(data, dict) else data   # signal tracks are keyed by chrom

def write_bedgraph(items, chrom, path, name):
    lines = [f'track type=bedGraph name="{name}"']
    lines += [f"{chrom}\t{it['start']}\t{it['end']}\t{it['value']}" for it in items]
    path.write_text("\n".join(lines) + "\n", newline="\n")

def write_bed(items, chrom, path, name):
    lines = [f'track name="{name}" itemRgb="On"']
    for it in items:
        s, e = it["chromStart"], it["chromEnd"]
        rgb = it.get("reserved") or it.get("itemRgb") or "0"
        lines.append(f"{chrom}\t{s}\t{e}\t{it.get('name', '')}\t0\t.\t{s}\t{e}\t{rgb}")
    path.write_text("\n".join(lines) + "\n", newline="\n")

# e.g. (chrom, start1, end1, OUT_DIR from your earlier cells)
write_bedgraph(ucsc_items("phastCons100way", chrom, start1, end1), chrom,
               OUT_DIR / "phastcons100.bedGraph", "PhastCons100")
write_bed(ucsc_items("encodeCcreCombined", chrom, start1, end1), chrom,
          OUT_DIR / "encode3_ccre.bed", "ENCODE3")